In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Find the directory where this notebook/script is running
current_dir = Path.cwd()

# Find the root project folder (hspital_readmission)
# If running inside 'notebooks', step up one level; otherwise use current directory
project_root = (
    current_dir.parent if current_dir.name == "notebooks" else current_dir
)

# Set the path directly to your .env file
env_path = project_root / ".env"

# Load the environment variables
load_dotenv(dotenv_path=env_path)

# Verify loading without printing raw passwords
db_user = os.getenv("DB_USER")

if db_user:
    print(
        f"✅ Environment variables loaded successfully from: {project_root.name}/.env"
    )
else:
    print(f"❌ Could not find .env file at: {env_path}")

✅ Environment variables loaded successfully from: hospital_readmission_project/.env


In [2]:
%%writefile config.py

import os
from dotenv import load_dotenv
from pathlib import Path

# Exact path matching your folder structure
env_path = Path(r"C:\Users\Administrator\Documents\analytics projects\hospital_readmission_project\.env")
load_dotenv(dotenv_path=env_path)

DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = os.getenv("DB_PORT", "5432")
DB_NAME     = os.getenv("DB_NAME")

missing = [k for k, v in {
    "DB_USER"    : DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
    "DB_NAME"    : DB_NAME
}.items() if v is None]

if missing:
    raise ValueError(
        f"These variables were not loaded from .env: {missing}\n"
        f"Check that your .env file exists at: {env_path}"
    )

DB_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

CONFIG = {
    "db_url"      : DB_URL,
    "raw_table"   : "diabetic_data",
    "clean_table" : "diabetic_data_clean",
    "schema"      : "public",
    "primary_key" : "encounter_id",
    "patient_key" : "patient_nbr",
    "target_col"  : "readmitted",
    "missing_sentinel" : "?",
    "numeric_cols": [
        "time_in_hospital", "num_lab_procedures", "num_procedures",
        "num_medications", "number_outpatient", "number_emergency",
        "number_inpatient", "number_diagnoses"
    ],
    "id_cols": [
        "admission_type_id", "discharge_disposition_id", "admission_source_id"
    ],
    "diag_cols": ["diag_1", "diag_2", "diag_3"],
    "medication_cols": [
        "metformin", "repaglinide", "nateglinide", "chlorpropamide",
        "glimepiride", "acetohexamide", "glipizide", "glyburide",
        "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
        "miglitol", "troglitazone", "tolazamide", "examide",
        "citoglipton", "insulin", "glyburide-metformin",
        "glipizide-metformin", "glimepiride-pioglitazone",
        "metformin-rosiglitazone", "metformin-pioglitazone"
    ],
    "categorical_cols": [
        "race", "gender", "age", "weight", "payer_code",
        "medical_specialty", "max_glu_serum", "A1Cresult"
    ],
    "binary_cols"         : ["change", "diabetesMed"],
    "expired_dispositions": [11, 13, 14, 19, 20, 21],
    "valid_gender"        : ["Male", "Female"],
    "valid_readmitted"    : ["<30", ">30", "NO"],
    "valid_med_values"    : ["No", "Steady", "Up", "Down"],
    "valid_change"        : ["Ch", "No"],
    "valid_diabetesMed"   : ["Yes", "No"],
    "polypharmacy_threshold": 10,
    "age_order": [
        "[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
        "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"
    ]
}

Overwriting config.py


In [3]:
%%writefile data_dictionary.py

import pandas as pd
import os

DATA_DICTIONARY = {
    "encounter_id": {
        "type"       : "Identifier",
        "description": "Unique identifier for each hospital encounter.",
        "notes"      : "Primary key. Must be unique across all rows."
    },
    "patient_nbr": {
        "type"       : "Identifier",
        "description": "Unique identifier for each patient. Repeats for multiple encounters.",
        "notes"      : "Used to link encounters for the same patient via window functions."
    },
    "race": {
        "type"       : "Categorical",
        "description": "Patient race.",
        "values"     : "AfricanAmerican, Asian, Caucasian, Hispanic, Other",
        "missing"    : "~2% encoded as ? — treated as Unknown. Never imputed."
    },
    "gender": {
        "type"       : "Categorical",
        "description": "Patient gender.",
        "values"     : "Male, Female, Unknown/Invalid",
        "notes"      : "Unknown/Invalid rows are flagged and excluded from analysis."
    },
    "age": {
        "type"       : "Ordinal",
        "description": "Patient age in 10-year bands e.g. [70-80).",
        "notes"      : "Midpoint extracted as age_midpoint feature for numeric analysis."
    },
    "weight": {
        "type"       : "Categorical",
        "description": "Patient weight in bands.",
        "missing"    : "~97% missing — too sparse to use. Presence flagged as weight_available."
    },
    "admission_type_id": {
        "type"       : "Categorical lookup",
        "description": "Type of hospital admission.",
        "values"     : "1=Emergency, 2=Urgent, 3=Elective, 4=Newborn, 6=NULL, 7=Trauma",
        "notes"      : "Stored as integer but is a category code. Cast to string in cleaning."
    },
    "discharge_disposition_id": {
        "type"       : "Categorical lookup",
        "description": "Where the patient was discharged to.",
        "values"     : "1=Home, 3=SNF, 6=Home with health service, 11=Expired etc.",
        "notes"      : "Codes 11,13,14,19,20,21 = death or hospice. Excluded from readmission rate."
    },
    "admission_source_id": {
        "type"       : "Categorical lookup",
        "description": "Where the patient was admitted from.",
        "values"     : "1=Physician referral, 7=Emergency room, 4=Transfer from hospital etc.",
        "notes"      : "Cast to string in cleaning."
    },
    "time_in_hospital": {
        "type"       : "Numeric",
        "description": "Number of days the patient stayed in hospital.",
        "range"      : "1 to 14",
        "notes"      : "Length of stay. Key clinical metric in every KPI view."
    },
    "payer_code": {
        "type"       : "Categorical",
        "description": "Insurance payer code.",
        "missing"    : "~40% missing. Filled with Unknown."
    },
    "medical_specialty": {
        "type"       : "Categorical",
        "description": "Medical specialty of the admitting physician.",
        "missing"    : "~49% missing. Filled with Unknown. Filtered from specialty KPIs."
    },
    "num_lab_procedures": {
        "type"       : "Numeric",
        "description": "Number of lab tests performed during the encounter."
    },
    "num_procedures": {
        "type"       : "Numeric",
        "description": "Number of non-lab procedures performed."
    },
    "num_medications": {
        "type"       : "Numeric",
        "description": "Total number of distinct medications administered.",
        "notes"      : "Used in polypharmacy flag — threshold is >= 10."
    },
    "number_outpatient": {
        "type"       : "Numeric",
        "description": "Number of outpatient visits in the year before this encounter."
    },
    "number_emergency": {
        "type"       : "Numeric",
        "description": "Number of emergency visits in the year before this encounter."
    },
    "number_inpatient": {
        "type"       : "Numeric",
        "description": "Number of inpatient visits in the year before this encounter.",
        "notes"      : "Strong predictor of readmission risk."
    },
    "diag_1": {
        "type"       : "ICD-9 code",
        "description": "Primary diagnosis code.",
        "notes"      : "Can be numeric, V-code, or E-code. Regex guard required before SQL CAST."
    },
    "diag_2": {
        "type"       : "ICD-9 code",
        "description": "Secondary diagnosis code.",
        "missing"    : "Legitimately absent for simpler cases. Filled with None."
    },
    "diag_3": {
        "type"       : "ICD-9 code",
        "description": "Tertiary diagnosis code.",
        "missing"    : "Legitimately absent. Filled with None."
    },
    "number_diagnoses": {
        "type"       : "Numeric",
        "description": "Total number of diagnoses entered for this encounter."
    },
    "max_glu_serum": {
        "type"       : "Categorical",
        "description": "Result of glucose serum test.",
        "values"     : ">200, >300, Norm, None",
        "notes"      : "None means the test was NOT performed — not a normal result."
    },
    "A1Cresult": {
        "type"       : "Categorical",
        "description": "Result of HbA1c test.",
        "values"     : ">7=poor control, >8=very poor control, Norm=good control, None=not tested",
        "notes"      : "None means test was not performed. Critical diabetes management indicator."
    },
    "insulin": {
        "type"       : "Categorical",
        "description": "Whether insulin was prescribed and whether dose was changed.",
        "values"     : "No, Steady, Up, Down"
    },
    "change": {
        "type"       : "Binary categorical",
        "description": "Whether any diabetes medication dose was changed during the encounter.",
        "values"     : "Ch=changed, No=not changed"
    },
    "diabetesMed": {
        "type"       : "Binary categorical",
        "description": "Whether any diabetes medication was prescribed.",
        "values"     : "Yes, No"
    },
    "readmitted": {
        "type"       : "Target variable",
        "description": "FORWARD-LOOKING: After this discharge, was the patient readmitted?",
        "values"     : "<30=within 30 days, >30=after 30 days, NO=not readmitted",
        "notes"      : "Does NOT identify the current encounter as a readmission. That is captured by is_itself_a_readmission via LAG() window function."
    }
}

def save_data_dictionary(dictionary, filepath="outputs/data_dictionary.csv"):
    os.makedirs("outputs", exist_ok=True)
    rows = []
    for col, meta in dictionary.items():
        row = {"column": col}
        row.update(meta)
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(filepath, index=False)
    print(f"Data dictionary saved to {filepath}")
    print(f"Columns documented: {len(rows)}")
    return df

if __name__ == "__main__":
    dd_df = save_data_dictionary(DATA_DICTIONARY)
    print(dd_df[["column", "type", "description"]].to_string(index=False))

Overwriting data_dictionary.py


In [4]:
%%writefile project_brief.md

# Hospital Readmission Analytics — Project Brief

## Background
Hospital readmissions within 30 days of discharge are a significant quality
and cost indicator in healthcare systems. Understanding which patient groups
and clinical factors drive readmissions allows hospitals to target
interventions more effectively.

## Dataset
Source: UCI Machine Learning Repository — Diabetes 130-US Hospitals (1999-2008)
Size: ~101,766 encounters across 130 US hospitals
Target variable: readmitted (<30, >30, NO)

## Analytical Questions

Q1.  What is the overall 30-day readmission rate across this hospital system?
Q2.  Which patient demographics carry the highest readmission risk?
Q3.  Which primary diagnosis categories drive the most readmissions?
Q4.  Which clinical departments have the highest readmission rates?
Q5.  Does medication management correlate with readmission risk?
Q6.  Do patients with prior inpatient or emergency visits have higher rates?
Q7.  Does length of stay or polypharmacy correlate with readmission?
Q8.  Which discharge disposition carries the highest readmission risk?
Q9.  Do repeat patients have higher readmission rates than first-time patients?
Q10. What is the clinical profile of the highest-risk patient segment?

## Expected Outputs
- Cleaned PostgreSQL table: diabetic_data_clean
- 8 KPI views: vw_kpi_*
- Statistical validation of key EDA findings
- 5-page Power BI dashboard with drill-through
- GitHub repository with full README

## Analytical Boundaries
- No predictive modeling — descriptive analytics only
- No causal inference — associations do not imply causation
- weight column excluded — ~97% missing
- Confounders not controlled

Overwriting project_brief.md
